# Sábado en Kaggle: un rango de las 992 → `submissions.jsonl`

Corre un rango de las 992 preguntas con el sistema final: **Gemma 4 E4B, temperatura 0,
verificación independiente en selección múltiple (C-25)** y **el índice construido en Turing**
(idéntico en todas las máquinas). Solo genera respuestas y entrega `<TAG>_submissions.jsonl`.

**Tres archivos de entrada** (nombres exactos):
- `paquete_colab.zip`: código (el de `dist/` del computador del equipo).
- `build_indice_nuevo.zip`: índice construido en Turing.
- `test_992.jsonl`: las 992 preguntas.

**Pasos:** cuenta **verificada con teléfono**; *Datasets → New Dataset* (privado) con los tres
archivos; *Code → New Notebook → File → Import Notebook* → este archivo; panel derecho:
**GPU T4 x2**, **Internet On**, *Add Input* → el Dataset; poner `TAG` y `RANGO` en la celda 2;
*Run All*. Al final: enlace a `<TAG>_submissions.jsonl` (también en el panel *Output*).
Kaggle puede descomprimir los zip del Dataset: el cuaderno funciona en ambos casos.

In [ ]:
# 1. GPU (deben aparecer las T4)
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 2. Parámetros y archivos

In [ ]:
# ===== AJUSTAR EN CADA MÁQUINA =====
TAG = "sabado_1"       # sabado_1, sabado_2, …: uno distinto por máquina
RANGO = "1 165"        # posiciones del archivo de preguntas, ambos extremos incluidos
SHA_FAISS = ""         # opcional: sha256 de index.faiss que imprimió Turing (se comprueba si se pone)
# ====================================
INICIO, FIN = map(int, RANGO.split())

import glob, hashlib, os, shutil, zipfile

def mas_reciente(patrones):
    c = [p for pat in patrones for p in glob.glob(pat, recursive=True)]
    return max(c, key=os.path.getmtime) if c else None

def sha256(ruta):
    h = hashlib.sha256()
    with open(ruta, "rb") as f:
        for b in iter(lambda: f.read(1 << 20), b""):
            h.update(b)
    return h.hexdigest()

PROYECTO = "/kaggle/working/proyecto"
shutil.rmtree(PROYECTO, ignore_errors=True); os.makedirs(PROYECTO)

# Código (paquete_colab.zip, con el C-25): zip, o carpeta si la plataforma lo descomprimió.
paq = mas_reciente([f"/kaggle/input/**/paquete_colab.zip", f"/kaggle/input/**/paquete_colab (*).zip"])
if paq:
    zipfile.ZipFile(paq).extractall(PROYECTO); print("Código:", paq)
else:
    conf = mas_reciente([f"/kaggle/input/**/src/config.py"])
    assert conf, "No encuentro paquete_colab.zip"
    shutil.copytree(os.path.dirname(os.path.dirname(conf)), PROYECTO, dirs_exist_ok=True); print("Código:", conf)

# Índice construido en Turing (build_indice_nuevo.zip): reemplaza build/indice del paquete.
shutil.rmtree(f"{PROYECTO}/build/indice", ignore_errors=True)
idx = mas_reciente([f"/kaggle/input/**/build_indice_nuevo*.zip"])
if idx:
    zipfile.ZipFile(idx).extractall(PROYECTO); print("Índice:", idx)
else:
    faiss_ = mas_reciente([f"/kaggle/input/**/build/indice/index.faiss"])
    assert faiss_, "No encuentro build_indice_nuevo.zip"
    shutil.copytree(os.path.dirname(faiss_), f"{PROYECTO}/build/indice"); print("Índice:", faiss_)

# Preguntas del sábado.
preg = mas_reciente([f"/kaggle/input/**/test_992*.jsonl"])
assert preg, "No encuentro test_992.jsonl"
os.makedirs(f"{PROYECTO}/data", exist_ok=True)
shutil.copy(preg, f"{PROYECTO}/data/test_992.jsonl")
n = sum(1 for l in open(f"{PROYECTO}/data/test_992.jsonl", encoding="utf-8") if l.strip())
print("Preguntas:", preg, "→", n)

# Esquema oficial de la entrega (el paquete de índice no lo trae): copia exacta del reto.
import json
os.makedirs(f"{PROYECTO}/schema", exist_ok=True)
ESQUEMA_OFICIAL = r"""{"$schema":"https://json-schema.org/draft/2020-12/schema","$id":"submission.schema.json","title":"Entrega de la Hackathon 2026 - Derecho colombiano","description":"Una linea JSON por item del test. Los nombres de campo replican las llaves de los prompts oficiales de la hackathon y son obligatorios: sobre ellos opera el evaluador.","type":"object","required":["id","formato","abstencion","pasajes_recuperados"],"properties":{"id":{"type":"integer","description":"Identificador del item, tal como viene en test_992.jsonl."},"formato":{"enum":["multiple_choice","semi_open","open_ended"]},"abstencion":{"type":"boolean","description":"true si el sistema declara no tener fundamento suficiente en su corpus. Se puntua: abstenerse bien suma, abstenerse donde el corpus si daba fundamento resta."},"pasajes_recuperados":{"type":"array","description":"Evidencia recuperada que sustenta la respuesta. Vacio solo si abstencion es true. Debe permitir reconstruir la respuesta en la verificacion en vivo del sabado.","items":{"type":"object","required":["doc_id","texto"],"properties":{"doc_id":{"type":"string","description":"Identificador del documento en el corpus del equipo (debe existir en corpus_manifest.json)."},"inicio":{"type":"integer","minimum":0,"description":"Offset de caracter donde inicia el pasaje en el documento."},"fin":{"type":"integer","minimum":0},"texto":{"type":"string","minLength":1,"description":"Texto literal del pasaje recuperado."},"score":{"type":"number","description":"Puntaje de similitud del recuperador."}}}},"latencia_ms":{"type":"integer","minimum":0,"description":"Opcional. Tiempo de la consulta de punta a punta."}},"allOf":[{"if":{"properties":{"formato":{"const":"multiple_choice"}},"required":["formato"]},"then":{"required":["respuesta_correcta","justificacion","descarte_opciones"],"properties":{"respuesta_correcta":{"enum":["A","B","C","D"],"description":"Etiqueta de la opcion elegida. Con abstencion true se admite null."},"justificacion":{"type":"string","description":"Debe citar la norma, articulo, ley o jurisprudencia aplicable: de aqui se extraen las citas que se puntuan."},"descarte_opciones":{"type":"object","description":"Letras de las opciones incorrectas como llaves, razon breve como valor."}}}},{"if":{"properties":{"formato":{"const":"semi_open"}},"required":["formato"]},"then":{"required":["respuesta","palabras_clave","referencia_legal"],"properties":{"respuesta":{"type":"string","description":"De 3 a 5 oraciones, maximo 150 palabras."},"palabras_clave":{"type":"array","items":{"type":"string"}},"referencia_legal":{"type":"string","description":"Norma o articulo citado."}}}},{"if":{"properties":{"formato":{"const":"open_ended"}},"required":["formato"]},"then":{"required":["marco_normativo","analisis","jurisprudencia","conclusion"],"properties":{"marco_normativo":{"type":"string"},"analisis":{"type":"string","description":"De 5 a 8 oraciones."},"jurisprudencia":{"type":"string"},"conclusion":{"type":"string"}}}}]}"""
json.dump(json.loads(ESQUEMA_OFICIAL), open(f"{PROYECTO}/schema/submission.schema.json", "w", encoding="utf-8"), ensure_ascii=False, indent=2)

h = sha256(f"{PROYECTO}/build/indice/index.faiss")
print("sha256 index.faiss:", h)
if SHA_FAISS:
    assert h == SHA_FAISS.strip().lower(), "¡El índice NO es el de Turing! Revise build_indice_nuevo.zip"
    print("Índice idéntico al de Turing ✓")
print(open(f"{PROYECTO}/build/indice/index_manifest.json", encoding="utf-8").read()[:300])

## 3–4. Dependencias y GPU

In [ ]:
# 3. Dependencias (≈ 3–5 min). Se conserva el torch con CUDA del entorno.
# pymupdf: lo usa la extracción de PDF del pipeline.
!pip install -q sentence-transformers==6.1.0 transformers==5.17.0 faiss-cpu==1.15.1 bm25s==0.3.11 PyStemmer==3.1.0 python-dotenv==1.2.3 jsonschema==4.26.0 pymupdf==1.28.2 httpx huggingface_hub
!pip install -q llama-cpp-python==0.3.35 --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu124 --only-binary llama-cpp-python || echo "SIN RUEDA PRECOMPILADA"
import subprocess, sys
if subprocess.run([sys.executable, "-m", "pip", "show", "llama-cpp-python"], capture_output=True).returncode != 0:
    print("Compilando llama-cpp-python con CUDA (≈ 15 min)…")
    !CMAKE_ARGS="-DGGML_CUDA=on" pip install -q llama-cpp-python==0.3.35 --no-binary llama-cpp-python --force-reinstall --no-deps

In [ ]:
# 4. llama.cpp en la GPU (debe decir True). Si el entorno trae CUDA 13, se instalan las librerías
# de CUDA 12 como paquetes de Python y se apunta a ellas; los !python heredan la ruta.
import ctypes, glob, os, subprocess, sys
prueba = subprocess.run([sys.executable, "-c", "import llama_cpp"], capture_output=True, text=True)
if prueba.returncode != 0 and ".so.12" in prueba.stderr:
    print("Faltan librerías de CUDA 12: instalándolas…")
    !pip install -q nvidia-cuda-runtime-cu12 nvidia-cublas-cu12
    libs = [d for pat in ("nvidia/cuda_runtime/lib", "nvidia/cublas/lib")
            for d in glob.glob(f"/usr/local/lib/python3*/dist-packages/{pat}")]
    os.environ["LD_LIBRARY_PATH"] = ":".join(libs + [os.environ.get("LD_LIBRARY_PATH", "")])
    for d in libs:
        for so in sorted(glob.glob(d + "/lib*.so.12")):
            ctypes.CDLL(so, mode=ctypes.RTLD_GLOBAL)
import llama_cpp
print("GPU disponible para llama.cpp:", llama_cpp.llama_supports_gpu_offload())

## 5. Configuración y prueba del decoder

In [ ]:
# 5. Configuración (temperatura 0, Gemma 4 E4B, C-25) y prueba del decoder en la GPU
%cd /kaggle/working/proyecto
import os
os.environ.update({"DECODER_MODEL": "google/gemma-4-E4B-it",
                   "DECODER_GGUF_REPO": "ggml-org/gemma-4-E4B-it-GGUF",
                   "DECODER_GGUF_FILE": "gemma-4-E4B-it-Q8_0.gguf",
                   "DECODER_BACKEND": "llamacpp", "ENCODER_BATCH_SIZE": "64",
                   "CORPUS_SOURCE": "local", "PYTHONIOENCODING": "utf-8",
                   "CUDA_VISIBLE_DEVICES": "0",  # Kaggle trae dos T4: se usa una
                   # JAX/TF preinstalados reservan el 75 % de la VRAM si se importan: a CPU.
                   "JAX_PLATFORMS": "cpu", "XLA_PYTHON_CLIENT_PREALLOCATE": "false",
                   "TF_FORCE_GPU_ALLOW_GROWTH": "true", "USE_TF": "0", "USE_JAX": "0"})
!python -c "from src.config import config; config.validar_final(); print('Decoder:', config.decoder_gguf_file, '| temperatura:', config.decoder_temperature, '| C-25 verificación MC:', config.mc_verificacion_independiente)"
!ENCODER_DEVICE=cuda DECODER_DEVICE=cuda python -m src.generation.ping --sin-cache

## 6. Responder el rango

In [ ]:
# 6. Responder el rango (si se interrumpe con la sesión viva, vuelva a ejecutar esta celda: sigue
# donde quedó). Verifica que el índice corresponda a su manifiesto (sha256), como run.py --indice-existente.
%cd /kaggle/working/proyecto
import json, hashlib
def _sha(p):
    h = hashlib.sha256()
    with open(p, "rb") as f:
        for b in iter(lambda: f.read(1 << 20), b""): h.update(b)
    return h.hexdigest()
m = json.load(open("build/indice/index_manifest.json", encoding="utf-8"))
assert m["sha256_faiss"] == _sha("build/indice/index.faiss") and m["sha256_chunks"] == _sha("build/indice/chunks.jsonl"), "El índice no corresponde a su manifiesto"
print("Índice verificado:", m["n_fragmentos"], "fragmentos")
!ENCODER_DEVICE=cuda DECODER_DEVICE=cuda python -m src.pipeline.main --split test --rango {INICIO} {FIN} --tag {TAG}; echo "CÓDIGO DE SALIDA: $? (0 = bien)"

## 7. Entregar

In [ ]:
# 7. Entregar el submissions.jsonl de este rango
import json, shutil
%cd /kaggle/working/proyecto
filas = [json.loads(l) for l in open(f"runs/{TAG}/submissions.jsonl", encoding="utf-8") if l.strip()]
total = sum(1 for l in open("data/test_992.jsonl", encoding="utf-8") if l.strip())
esperadas = min(FIN, total) - INICIO + 1
print(f"{len(filas)} respuestas (se esperaban {esperadas}); ids {filas[0]['id']} … {filas[-1]['id']}")
if len(filas) != esperadas:
    print("ATENCIÓN: faltan respuestas; vuelva a ejecutar la celda 6 (continúa donde quedó).")
destino = f"/kaggle/working/{TAG}_submissions.jsonl"
shutil.copy(f"runs/{TAG}/submissions.jsonl", destino)
%cd /kaggle/working
from IPython.display import FileLink
display(FileLink(f"{TAG}_submissions.jsonl"))